# IE643 • Latent Navigators — Novelty experiment
### Clean-Anchored Minority-Aware Robust Fine-Tuning (CAMARF)
**Ananna Biswas • 25M1727 • 2026 Novelty Assessment**

**Research hypothesis:** Keeping one **clean anchor** for each training image, limiting optimization of pretrained visual features, and allocating corrupted views toward rare classes / difficult illumination will improve robustness without damaging clean detection.

**This notebook is an experiment, not evidence of success until executed.** It runs three matched-budget detector arms and compares against your **saved original baseline and combined YOLOv8n checkpoints**:

- `uniform_views`: every training image appears once clean and once under a randomly selected degradation.
- `minority_sampling`: same clean anchors and the same number of degraded samples, but rare-class images are preferentially sampled for the degraded views.
- `minority_lowlight`: same sampling as above, with an additional low-light focus for minority images (**proposed novelty**).

Checkpoint selection uses only VOC **tuning images**, evaluating both clean and degraded views. VOC test, RTTS, DAWN, and ExDark are **held out** for final, one-time evaluation. The experiment explicitly reports any clean-image regression. It does **not** guarantee improvements.

**Google Colab:** Runtime → Change runtime type → **T4 GPU** (or a better GPU). Execute cells from top to bottom. Files save to `MyDrive/IE643_LatentNavigators/novelty_2026/` and are restart-safe where possible.

**Compute disclosure:** This chat environment has CPU only and does not contain your full VOC/RTTS/DAWN image sets, so the new detectors cannot be trained/evaluated here. When you run this notebook on Colab GPU, it generates real measured results rather than simulated numbers.


## 1 — Install packages, mount Drive, and configure experiment

In [ ]:
import os, sys, subprocess, importlib.util, json, random, shutil, zipfile, tarfile, math, re, hashlib, time
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

def ensure(pkg, module=None):
    if importlib.util.find_spec(module or pkg.split('=')[0].split('>')[0]) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
ensure('ultralytics>=8.3,<9','ultralytics')
ensure('pandas','pandas')
ensure('matplotlib','matplotlib')
import cv2, numpy as np, pandas as pd, yaml, torch, matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from ultralytics import YOLO
ImageFile.LOAD_TRUNCATED_IMAGES=True
cv2.setNumThreads(0)
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE=Path('/content/drive/MyDrive/IE643_LatentNavigators')
    LOCAL=Path('/content/ie643_novelty')
else:
    DRIVE=Path(os.getenv('IE643_PROJECT_DIR','./IE643_LatentNavigators')).resolve()
    LOCAL=Path(os.getenv('IE643_LOCAL_DIR','./ie643_novelty')).resolve()
CFG={
    'seed':0, 'tune_n':1000, 'imgsz':640, 'batch':16, 'epochs':8,
    'freeze':10, 'lr0':0.00020, 'lrf':0.20, 'warmup_epochs':1.0,
    'num_workers':2, 'clean_guard_drop':0.005,
    'syn_n':500, 'selection_n':1000, 'do_train':True,
    'run_arms':['uniform_views','minority_sampling','minority_lowlight'],
    'syn_types':['lowlight','haze','rain','snow','sandstorm'],
    'syn_severities':[1,2,3],
    'minority_weight':5.0,
    'minority_lowlight_probability':0.70,
    'max_side_train':1000,
    'source_voc':None, # optional extracted VOC2012 root with JPEGImages + Annotations
    'auto_download_voc':True,
    'rtts_source':'/content/drive/MyDrive/datasets/RTTS.zip',
    'dawn_source':'/content/drive/MyDrive/datasets/DAWN.zip',
    'exdark_source':None, # optional *extracted* ExDark folder, containing images + .txt ground truth
    'evaluate_real':True,
}
CLASSES=['person','car','chair','bottle','dog','bicycle','motorbike','bus','boat','cat']
MAJ=CLASSES[:5]; MIN=CLASSES[5:]; CIDX={c:i for i,c in enumerate(CLASSES)}
EXPERIMENT=DRIVE/'novelty_2026'; RUNS=EXPERIMENT/'yolo_runs'; OUT=EXPERIMENT/'results'
YDATA=LOCAL/'yolo_data'; LOCAL_DATA=LOCAL/'source_data'
for p in (EXPERIMENT,RUNS,OUT,YDATA,LOCAL_DATA): p.mkdir(parents=True,exist_ok=True)
CUDA=torch.cuda.is_available()
print('Device',torch.cuda.get_device_name(0) if CUDA else 'CPU - training will be slow')
print('Ultralytics version:', __import__('ultralytics').__version__)
print('Experiment results:',EXPERIMENT)
(OUT/'configuration.json').write_text(json.dumps(CFG,indent=2))

## 2 — Locate ORIGINAL checkpoints (automatically search Drive or upload missing best.pt)

If the original project folder is missing on Drive, this cell asks for your **baseline** `best.pt`, then your **combined** `best.pt`, and safely saves them under distinct names. **Do not upload the combined file as baseline.**


In [ ]:
# Safely locate your previously trained checkpoints.
# Important: files uploaded to a ChatGPT conversation are NOT automatically in Google Colab or Drive.
# The original YOLO training run had separate baseline and combined checkpoints, both named best.pt.

from pathlib import Path

BASELINE_OVERRIDE = None  # optional: e.g. '/content/drive/MyDrive/OtherFolder/baseline_best.pt'
COMBINED_OVERRIDE = None  # optional: e.g. '/content/drive/MyDrive/OtherFolder/combined_best.pt'

def _candidate_roots():
    roots = [DRIVE]
    if IN_COLAB:
        home = Path('/content/drive/MyDrive')
        try:
            for child in home.iterdir():
                if child.is_dir() and child != DRIVE and ('latent' in child.name.lower() or 'ie643' in child.name.lower()):
                    roots.append(child)
        except OSError:
            pass
    return roots

def locate_saved_checkpoint(role, override=None):
    if override:
        path = Path(override).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'The {role} override path does not exist: {path}')
        return path
    # Prefer the known paths from your original run configuration.
    for project_root in _candidate_roots():
        paths = [project_root / 'checkpoints' / f'det_{role}.pt',
                 project_root / 'yolo_runs' / role / 'weights' / 'best.pt',
                 project_root / 'yolo_runs' / role / 'best.pt']
        for path in paths:
            if path.is_file():
                print(f'Found {role}: {path}')
                return path
        # Look only under the specific role, not every .pt on Drive.
        run_folder = project_root / 'yolo_runs' / role
        if run_folder.is_dir():
            matches = sorted(run_folder.rglob('best.pt'))
            if len(matches) == 1:
                print(f'Found {role}: {matches[0]}')
                return matches[0]
    return None

def upload_missing_checkpoint(role):
    if not IN_COLAB:
        raise FileNotFoundError(f'Missing {role} checkpoint. Set the corresponding *_OVERRIDE path.')
    from google.colab import files
    print(f'\n{role.upper()} CHECKPOINT NOT FOUND on the mounted Google Drive.')
    print(f'Upload the {role.upper()} run best.pt saved on your computer (NOT the other run).')
    print('Upload only one checkpoint in this dialog. Your file may be named best.pt.')
    uploaded = files.upload()
    if not uploaded:
        if role == 'combined':
            print('Combined upload skipped; its reference evaluation will be unavailable until added.')
            return None
        raise FileNotFoundError('Baseline checkpoint is required. Re-run this cell and upload BASELINE best.pt.')
    if len(uploaded) != 1:
        raise ValueError(f'Upload exactly one {role} .pt checkpoint at a time.')
    name, raw = next(iter(uploaded.items()))
    if not name.lower().endswith('.pt'):
        raise ValueError(f'Expected a .pt checkpoint, received {name!r}')
    folder = DRIVE / 'checkpoints'
    folder.mkdir(parents=True, exist_ok=True)
    dest = folder / f'det_{role}.pt'
    dest.write_bytes(raw)
    print(f'Saved {role} checkpoint on Drive: {dest} ({len(raw):,} bytes)')
    return dest

ORIGINAL_BASE = locate_saved_checkpoint('baseline', BASELINE_OVERRIDE)
if ORIGINAL_BASE is None:
    ORIGINAL_BASE = upload_missing_checkpoint('baseline')
ORIGINAL_COMB = locate_saved_checkpoint('combined', COMBINED_OVERRIDE)
if ORIGINAL_COMB is None:
    ORIGINAL_COMB = upload_missing_checkpoint('combined')

def check_names(path, role):
    if path is None:
        return
    names = YOLO(str(path)).names
    actual = [names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)
    assert actual == CLASSES, (f'Incorrect {role} checkpoint: expected classes {CLASSES}, got {actual}. '
                               'Please verify you uploaded the right 10-class model.')
    print(f'✔ {role} checkpoint loaded; 10-class mapping correct: {path}')

check_names(ORIGINAL_BASE, 'baseline')
check_names(ORIGINAL_COMB, 'combined')
print('Checkpoint setup complete. Proceed to the next cell.')


## 3 — Reconstruct the EXACT prior VOC train/tune/test split

In [ ]:
VOC20=['aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair','cow','diningtable',
       'dog','horse','motorbike','person','pottedplant','sheep','sofa','train','tvmonitor']
NAME_MAP={c:c for c in VOC20}
NAME_MAP.update({'people':'person','pedestrian':'person','persons':'person','human':'person',
                 'motorcycle':'motorbike','motor':'motorbike','airplane':'aeroplane','plane':'aeroplane'})

def parse_voc_xml(xml_path):
    root=ET.parse(xml_path).getroot(); size=root.find('size')
    w=h=None
    if size is not None:
        w=int(float(size.findtext('width') or '0')) or None
        h=int(float(size.findtext('height') or '0')) or None
    objs=[]
    for o in root.findall('object'):
        b=o.find('bndbox')
        if b is None: continue
        try: coords=[float(b.findtext(k)) for k in ('xmin','ymin','xmax','ymax')]
        except (ValueError,TypeError):continue
        name=NAME_MAP.get((o.findtext('name') or '').strip().lower())
        if name is None:continue
        objs.append((name,*coords,int(o.findtext('difficult') or '0')))
    return dict(w=w,h=h,objs=objs,labels=sorted({x[0] for x in objs}))

def get_voc():
    opts=[CFG['source_voc'], LOCAL_DATA/'VOCdevkit/VOC2012',
          DRIVE/'datasets/VOCdevkit/VOC2012',
          DRIVE/'data_cache/VOCdevkit/VOC2012',
          Path('/content/data/VOCdevkit/VOC2012')]
    for op in opts:
        if op and (Path(op)/'Annotations').is_dir() and (Path(op)/'ImageSets/Main/train.txt').exists():
            print('Using VOC:',op); return Path(op)
    if not CFG['auto_download_voc']:
        raise FileNotFoundError('PASCAL VOC2012 not found. Set CFG["source_voc"] or enable auto_download_voc.')
    print('Downloading PASCAL VOC 2012 trainval; this may use ~2GB. Only done if not already present.')
    try:
        import torchvision
        torchvision.datasets.VOCDetection(root=str(LOCAL_DATA),year='2012',image_set='trainval',download=True)
    except Exception as err:
        print('Primary VOC download unavailable; trying the project notebook mirror:',err)
        import urllib.request
        tar=LOCAL_DATA/'VOCtrainval_11-May-2012.tar'
        if not tar.exists():
            urllib.request.urlretrieve('https://pjreddie.com/media/files/VOCtrainval_11-May-2012.tar',tar)
        with tarfile.open(tar) as t:t.extractall(LOCAL_DATA,filter='data')
    p=LOCAL_DATA/'VOCdevkit/VOC2012'
    assert (p/'Annotations').exists();return p

VOC=get_voc()
def records(split):
    ids=(VOC/'ImageSets/Main'/f'{split}.txt').read_text().split()
    recs=[]
    for n in tqdm(ids,desc=f'VOC {split}'):
        r=parse_voc_xml(VOC/'Annotations'/f'{n}.xml')
        r.update(path=str(VOC/'JPEGImages'/f'{n}.jpg'),id=n,source='voc')
        recs.append(r)
    return recs
train_pool=records('train'); val_pool=records('val')

def imbalanced_indices(recs,seed=0):
    # Exact sampler from original IE643_LatentNavigators notebook, Section 3.
    rng=random.Random(seed); chosen=set();selected=[];counts=Counter()
    minority_set=set(MIN)
    by={c:[i for i,r in enumerate(recs) if c in r['labels']] for c in CLASSES}
    def add(i):
        chosen.add(i);selected.append(i);counts.update(set(recs[i]['labels'])&set(CLASSES))
    for c in sorted(MIN,key=lambda c:len(by[c])):
        pool=[i for i in by[c] if i not in chosen];rng.shuffle(pool)
        for i in pool:
            if counts[c]>=30:break
            others=(set(recs[i]['labels'])&minority_set)-{c}
            if any(counts[o]>=30 for o in others):continue
            add(i)
    for c in sorted(MAJ,key=lambda c:len(by[c])):
        pool=[i for i in by[c] if i not in chosen and not(set(recs[i]['labels'])&minority_set)]
        rng.shuffle(pool)
        for i in pool:
            if counts[c]>=300:break
            add(i)
    return sorted(selected)
train_recs=[train_pool[i] for i in imbalanced_indices(train_pool,CFG['seed'])]
permutation=list(range(len(val_pool)));random.Random(CFG['seed']).shuffle(permutation)
tune_recs=[val_pool[i] for i in permutation[:CFG['tune_n']]]
test_recs=[val_pool[i] for i in permutation[CFG['tune_n']:]]
print('Train',len(train_recs),'Tune',len(tune_recs),'Test',len(test_recs))
counts=Counter(c for r in train_recs for c in r['labels'] if c in CIDX)
audit=pd.DataFrame({'class':CLASSES,'training_positive_images':[counts[c] for c in CLASSES]})
print(audit.to_string(index=False))
assert len(train_recs)==1176, 'Training subset differs from prior experiment. Investigate before training!'
assert [counts[c] for c in MIN]==[30]*5, 'Minority count differs: stop and inspect dataset or sampler.'
assert not ({r['id'] for r in train_recs}&{r['id'] for r in tune_recs+test_recs}), 'Leakage!'
assert not ({r['id'] for r in tune_recs}&{r['id'] for r in test_recs}), 'Tune/test overlap!'
audit.to_csv(OUT/'train_class_audit.csv',index=False)

## 4 — Reuse original weather simulation to keep tests comparable

In [ ]:
CFG['degr_types']=['haze','lowlight','snow','rain','sandstorm']
def _lerp(sev, lo, hi):
    a = (float(np.clip(sev, 1, 3)) - 1) / 2
    return lo + a * (hi - lo)


def fractal_noise(h, w, rng, octaves=5, base=3):
    """Multi-octave value noise in [0, 1]: smooth, patchy structure at several scales."""
    out = np.zeros((h, w), np.float32)
    amp, total = 1.0, 0.0
    for o in range(octaves):
        cells = base * 2 ** o
        gh, gw = max(2, int(round(cells * h / max(h, w))) + 1), max(2, int(round(cells * w / max(h, w))) + 1)
        grid = rng.random((gh, gw)).astype(np.float32)
        out += amp * cv2.resize(grid, (w, h), interpolation=cv2.INTER_CUBIC)
        total += amp
        amp *= 0.5
    out /= total
    return (out - out.min()) / (out.max() - out.min() + 1e-6)


def pseudo_depth(h, w, rng):
    """Rough scene depth in [0, 1] (1 = far): far above a random tilted horizon, nearer toward the bottom."""
    y = np.linspace(0, 1, h, dtype=np.float32)[:, None]
    x = np.linspace(-1, 1, w, dtype=np.float32)[None, :]
    horizon = rng.uniform(0.25, 0.6) + rng.uniform(-0.12, 0.12) * x
    d = 1.0 - 0.8 * np.clip((y - horizon) / (1.0 - horizon + 1e-6), 0, 1)
    return np.clip(0.75 * d + 0.25 * fractal_noise(h, w, rng, octaves=3), 0, 1)


def _motion_kernel(length, angle_deg):
    """Line kernel of the given length, rotated by angle_deg from vertical."""
    length = max(3, int(length) | 1)
    k = np.zeros((length, length), np.float32)
    k[:, length // 2] = 1.0
    M = cv2.getRotationMatrix2D(((length - 1) / 2, (length - 1) / 2), angle_deg, 1.0)
    k = cv2.warpAffine(k, M, (length, length))
    return k / (k.sum() + 1e-6)


def _to_float(img):
    return img.astype(np.float32) / 255.0


def _to_uint8(x):
    return (np.clip(x, 0, 1) * 255 + 0.5).astype(np.uint8)


def _sensor(x, rng, blur_sigma, noise_std):
    if blur_sigma > 0.05:
        x = cv2.GaussianBlur(x, (0, 0), blur_sigma)
    return x + rng.normal(0, noise_std, x.shape).astype(np.float32)


def deg_haze(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    d = pseudo_depth(h, w, rng)
    hetero = 0.7 + 0.6 * fractal_noise(h, w, rng)
    beta = _lerp(sev, 0.6, 2.8) * rng.uniform(0.85, 1.15)
    A = np.clip(rng.uniform(0.75, 0.95) + rng.uniform(-0.03, 0.03, 3), 0, 1).astype(np.float32)
    t = np.exp(-beta * d * hetero)[..., None]
    x = x * t + A * (1 - t)
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.0, 0.8), 0.008))


def deg_lowlight(img, sev, rng):
    lin = _to_float(img) ** 2.2
    lin = lin * _lerp(sev, 0.25, 0.04) * rng.uniform(0.8, 1.2)
    lin = lin * (1 + rng.uniform(-0.15, 0.15, 3)).astype(np.float32)          # white-balance shift
    photons = _lerp(sev, 300.0, 40.0)                                         # fewer photons = more shot noise
    lin = rng.poisson(np.clip(lin, 0, 1) * photons).astype(np.float32) / photons
    lin = lin + rng.normal(0, _lerp(sev, 0.002, 0.01), lin.shape).astype(np.float32)
    return _to_uint8(np.clip(lin, 0, 1) ** (1 / 2.2))


def deg_snow(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    veil = _lerp(sev, 0.12, 0.45)
    gray = x.mean(axis=2, keepdims=True)
    x = x * (1 - 0.4 * veil) + gray * 0.4 * veil
    x = x * (1 - veil) + 0.9 * veil
    angle = rng.uniform(-30, 30)
    scale = max(h, w) / 500
    for density, size, blur_len, opacity in [(_lerp(sev, 0.004, 0.02), 1, 3, 0.75),
                                             (_lerp(sev, 0.0008, 0.005), 3, 7, 0.9)]:
        m = (rng.random((h, w)) < density).astype(np.float32)
        k = max(1, int(round(size * scale)))
        if k > 1:
            m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        m = cv2.filter2D(m, -1, _motion_kernel(blur_len * scale, angle))
        m = cv2.GaussianBlur(m, (0, 0), 0.6)
        m = np.clip(m / (m.max() + 1e-6) * opacity * 1.5, 0, opacity)[..., None]
        x = x * (1 - m) + m
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.0, 0.6), 0.005))


def deg_rain(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    scale = max(h, w) / 500
    drops = (rng.random((h, w)) < _lerp(sev, 0.002, 0.008)).astype(np.float32)
    streaks = cv2.filter2D(drops, -1, _motion_kernel(_lerp(sev, 12, 35) * scale, rng.uniform(-20, 20)))
    streaks = cv2.GaussianBlur(streaks, (0, 0), 0.7)
    streaks = np.clip(streaks / (streaks.max() + 1e-6), 0, 1)[..., None]
    alpha = _lerp(sev, 0.35, 0.7)
    x = x * _lerp(sev, 0.95, 0.75)
    x = x * (1 - alpha * streaks) + alpha * streaks * 0.85
    veil = _lerp(sev, 0.05, 0.2)
    x = x * (1 - veil) + veil * 0.7
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.3, 1.2), 0.006))


def deg_sandstorm(img, sev, rng):
    x = _to_float(img)
    h, w = x.shape[:2]
    dust = np.clip(np.array([0.80, 0.62, 0.40]) * rng.uniform(0.9, 1.1, 3), 0, 1).astype(np.float32)
    d = pseudo_depth(h, w, rng)
    hetero = 0.6 + 0.8 * fractal_noise(h, w, rng)
    t = np.exp(-_lerp(sev, 0.8, 3.2) * rng.uniform(0.85, 1.15) * d * hetero)[..., None]
    x = x * t + dust * (1 - t)
    x[..., 2] *= _lerp(sev, 0.9, 0.6)                                           # blue absorption, yellow cast
    grains = cv2.GaussianBlur((rng.random((h, w)) < _lerp(sev, 0.002, 0.01)).astype(np.float32), (0, 0), 0.8)
    grains = np.clip(grains / (grains.max() + 1e-6), 0, 1)[..., None] * 0.5
    x = x * (1 - grains) + dust * grains
    return _to_uint8(_sensor(x, rng, _lerp(sev, 0.3, 1.0), 0.008))


DEGRADATIONS = {'haze': deg_haze, 'lowlight': deg_lowlight, 'snow': deg_snow, 'rain': deg_rain,
                'sandstorm': deg_sandstorm}
DEG_ID = {k: i + 1 for i, k in enumerate(DEGRADATIONS)}


def apply_degradation(img, kind, sev, rng):
    if kind in (None, 'none', 'clean'):
        return img
    return DEGRADATIONS[kind](img, sev, rng)


def random_degradation(img, rng, p=1.0, kinds=None):
    """Training augmentation: with probability p, a random type at a random continuous severity in [1, 3]."""
    if rng.random() >= p:
        return img
    kinds = kinds or CFG['degr_types']
    return apply_degradation(img, kinds[rng.integers(len(kinds))], rng.uniform(1.0, 3.0), rng)


def fixed_rng(seed, i, kind='clean', sev=0):
    """Reproducible generator for evaluation: the same image always gets the same degradation."""
    return np.random.default_rng([int(seed), int(i), DEG_ID.get(kind, 0), int(round(sev * 10))])

## 5 — Build equal-sized training sets with mandatory clean anchors

In [ ]:
def image_size(rec):
    if rec.get('w') and rec.get('h'):return int(rec['w']),int(rec['h'])
    with Image.open(rec['path']) as im: w,h=im.size
    rec['w'],rec['h']=w,h
    return w,h

def yolo_label_lines(rec):
    w,h=image_size(rec);lines=[]
    for c,x1,y1,x2,y2,difficult in rec['objs']:
        if c not in CIDX or difficult:continue
        x1,x2=sorted((min(max(x1,0),w),min(max(x2,0),w)))
        y1,y2=sorted((min(max(y1,0),h),min(max(y2,0),h)))
        if x2-x1 < 1 or y2-y1 < 1:continue
        lines.append(f'{CIDX[c]} {(x1+x2)/(2*w):.6f} {(y1+y2)/(2*h):.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}')
    return '\n'.join(lines)

def export_imageset(name,items,overwrite=False):
    """items: [(record, None | (kind, severity, random_seed))]. No box geometry changes."""
    dst=YDATA/name
    marker=dst/'.done'
    if marker.exists() and not overwrite:return dst
    if dst.exists():shutil.rmtree(dst)
    img_dir=dst/'images';label_dir=dst/'labels'
    img_dir.mkdir(parents=True);label_dir.mkdir(parents=True)
    meta=[]
    for i,(rec,degrade) in enumerate(tqdm(items,desc=f'Creating {name}')):
        src=Path(rec['path']);stem=f'{i:06d}';dest=img_dir/(stem+('.jpg' if degrade else src.suffix.lower()))
        if degrade is None:
            # Symlink clean image to avoid needless multi-GB duplicates.
            dest.symlink_to(src.resolve())
        else:
            kind,sev,random_seed=degrade
            with Image.open(src) as im: arr=np.asarray(im.convert('RGB'))
            after=apply_degradation(arr,kind,sev,np.random.default_rng(random_seed))
            Image.fromarray(after).save(dest,'JPEG',quality=92)
        (label_dir/f'{stem}.txt').write_text(yolo_label_lines(rec))
        meta.append(dict(file=str(dest),source=rec.get('id',src.stem),kind='clean' if degrade is None else degrade[0],
                         severity=0 if degrade is None else degrade[1],minority=bool(set(rec['labels'])&set(MIN))))
    pd.DataFrame(meta).to_csv(dst/'generation_manifest.csv',index=False)
    marker.touch()
    return dst

def train_items(arm):
    rng=np.random.default_rng(105)
    clean=[(r,None) for r in train_recs]
    if arm=='uniform_views':
        sampled=np.arange(len(train_recs))
    else:
        importance=np.array([1.0+CFG['minority_weight']*(bool(set(r['labels'])&set(MIN))) for r in train_recs],float)
        p=importance/importance.sum()
        sampled=rng.choice(len(train_recs),size=len(train_recs),replace=True,p=p)
    aug=[]
    for i in sampled:
        rec=train_recs[int(i)]
        if arm=='minority_lowlight' and (set(rec['labels'])&set(MIN)) and rng.random()<CFG['minority_lowlight_probability']:
            kind='lowlight'; sev=float(rng.uniform(1.7,3.0))
        else:
            kind=CFG['degr_types'][int(rng.integers(5))];sev=float(rng.uniform(1.0,2.6))
        aug.append((rec,(kind,sev,int(rng.integers(1,2**31-1)))))
    return clean+aug

MANIFEST_SUMMARY=[]
for arm in CFG['run_arms']:
    items=train_items(arm)
    assert len(items)==2352 and all(deg is None for _,deg in items[:1176])
    export_imageset(f'train_{arm}',items)
    met=pd.read_csv(YDATA/f'train_{arm}/generation_manifest.csv')
    MANIFEST_SUMMARY.append(dict(arm=arm,images=len(met),clean=int((met.kind=='clean').sum()),
                                 degraded=int((met.kind!='clean').sum()),
                                 minority_degraded=int(((met.kind!='clean')&met.minority).sum()),
                                 lowlight_degraded=int((met.kind=='lowlight').sum())))
print(pd.DataFrame(MANIFEST_SUMMARY).to_string(index=False))
pd.DataFrame(MANIFEST_SUMMARY).to_csv(OUT/'training_manifest_summary.csv',index=False)

## 6 — Prepare tuning datasets (kept completely out of training)

In [ ]:
export_imageset('voc_tune',[(r,None) for r in tune_recs])
# For tuning we use the same fixed source images, with deterministic corruptions.
# Never select models on the VOC test, RTTS, DAWN, or ExDark sets.
SELECTION=tune_recs[:min(CFG['selection_n'],len(tune_recs))]
for kind,sev in [('lowlight',2),('lowlight',3),('haze',3)]:
    name=f'tune_{kind}_s{sev}'
    items=[(r,(kind,sev,[CFG['seed'],i,DEG_ID[kind],sev*10])) for i,r in enumerate(SELECTION)]
    export_imageset(name,items)

def data_yaml(train_set,test_set):
    fp=LOCAL/f'{train_set}--{test_set}.yaml'
    data={'path':str(YDATA), 'train':f'{train_set}/images',
          'val':'voc_tune/images','test':f'{test_set}/images',
          'names':{i:c for i,c in enumerate(CLASSES)}}
    with fp.open('w') as f: yaml.safe_dump(data,f,sort_keys=False)
    return str(fp)

print('Prepared tune images:',len(tune_recs),'and shifted tuning subsets:',len(SELECTION))

## 7 — Train matched-budget YOLOv8n arms

All three experimental arms **start from the same saved original baseline detector** and use:
- the same 2,352 training instances (1,176 clean anchors + 1,176 degraded copies), batch size 16, 640px, 8 epochs, optimizer, and seed;
- frozen early backbone layers (`freeze=10`) and a small AdamW learning rate, to reduce catastrophic forgetting;
- no mosaic/mixup and modest color jitter so synthetic corrupted copies are not transformed unpredictably again.

The arms isolate the contributions of sampling and low-light targeting. The old baseline and old combined checkpoints remain **untouched**. If Colab disconnects, completed arms are skipped on rerun. Checkpoint selection occurs in the next cell.

In [ ]:
if CFG['do_train'] and not CUDA:
    raise RuntimeError('Change Colab runtime to T4 GPU before training. The setup and dataset stages work on CPU.')
TRAINING_SUMMARY=[]
for arm in CFG['run_arms']:
    run=RUNS/arm
    best=run/'weights/best.pt';last=run/'weights/last.pt'
    history_csv=run/'results.csv'
    completed=False
    if last.exists() and history_csv.exists():
        hist=pd.read_csv(history_csv)
        completed=len(hist)>=CFG['epochs']
    if completed and best.exists():
        print('Completed arm found:',arm,'— skipping retraining.')
    else:
        if last.exists():
            print('Found unfinished run:',arm,'— trying optimizer-state resume.')
            model=YOLO(str(last))
            model.train(resume=True)
            del model
        else:
            print('Fine-tuning',arm,'from',ORIGINAL_BASE)
            model=YOLO(str(ORIGINAL_BASE))
            model.train(data=data_yaml(f'train_{arm}','voc_tune'),
                    epochs=CFG['epochs'],batch=CFG['batch'],imgsz=CFG['imgsz'],
                    optimizer='AdamW',lr0=CFG['lr0'],lrf=CFG['lrf'],weight_decay=0.0005,
                    warmup_epochs=CFG['warmup_epochs'], freeze=CFG['freeze'],
                    seed=CFG['seed'],deterministic=True,workers=CFG['num_workers'],
                    mosaic=0.0,close_mosaic=0,mixup=0.0,copy_paste=0.0,
                    hsv_h=0.005,hsv_s=0.35,hsv_v=0.20,
                    fliplr=0.5,amp=True,cos_lr=True,patience=CFG['epochs']+2,
                    save_period=2,project=str(RUNS),name=arm,exist_ok=True,
                    plots=True,verbose=False,device=0)
            del model
    TRAINING_SUMMARY.append(dict(arm=arm,best_exists=best.exists(),last_exists=last.exists()))
print(pd.DataFrame(TRAINING_SUMMARY).to_string(index=False))
assert all(s['best_exists'] for s in TRAINING_SUMMARY),'At least one arm lacks best.pt: inspect logs before proceeding.'

## 8 — Evaluate and select checkpoint with a CLEAN safeguard; no test-set tuning

In [ ]:
def eval_one(ckpt,eval_set,train_set='train_uniform_views',split='test',saveplots=False):
    data=data_yaml(train_set,eval_set)
    model=YOLO(str(ckpt))
    r=model.val(data=data,split=split, imgsz=CFG['imgsz'],batch=32,
                conf=0.001,iou=0.60,plots=saveplots,verbose=False,
                device=0 if CUDA else 'cpu',workers=CFG['num_workers'],
                project=str(OUT/'val_runs'),name=f'{Path(ckpt).parent.parent.name}_{eval_set}',exist_ok=True)
    indices=list(map(int,r.box.ap_class_index))
    ap={CLASSES[i]:float(v) for i,v in zip(indices,r.box.ap50)}
    prec={CLASSES[i]:float(v) for i,v in zip(indices,r.box.p)}
    recall={CLASSES[i]:float(v) for i,v in zip(indices,r.box.r)}
    def group_avg(d,classes):
        a=[d[c] for c in classes if c in d]
        return float(np.mean(a)) if a else np.nan
    result={'clean_or_shifted':eval_set,'mAP50':float(r.box.map50),'mAP5095':float(r.box.map),
            'majority_AP50':group_avg(ap,MAJ),'minority_AP50':group_avg(ap,MIN),
            'majority_recall':group_avg(recall,MAJ),'minority_recall':group_avg(recall,MIN),
            'overall_recall':group_avg(recall,CLASSES),
            'overall_precision':group_avg(prec,CLASSES)}
    for c in CLASSES:
        result[f'ap50_{c}']=ap.get(c,np.nan)
        result[f'recall_{c}']=recall.get(c,np.nan)
    del model,r
    if CUDA:torch.cuda.empty_cache()
    return result

SELECTION_SETS=['voc_tune','tune_lowlight_s2','tune_lowlight_s3','tune_haze_s3']
# Candidates always evaluated on the SAME tune images, with a stable eval protocol.
candidate_paths={'original_baseline':ORIGINAL_BASE}
if ORIGINAL_COMB:candidate_paths['original_combined']=ORIGINAL_COMB
for arm in CFG['run_arms']:
    for key,fn in [('best','best.pt'),('last','last.pt')]:
        p=RUNS/arm/'weights'/fn
        if p.exists():candidate_paths[f'{arm}_{key}']=p

select_csv=OUT/'tune_candidate_metrics.csv'
prev=pd.read_csv(select_csv) if select_csv.exists() else pd.DataFrame()
known=set(zip(prev['candidate'],prev['clean_or_shifted'])) if len(prev) else set()
rows=[]
for cand,p in candidate_paths.items():
    for evalset in SELECTION_SETS:
        if (cand,evalset) in known:continue
        print('Tune',cand,'@',evalset)
        z=eval_one(p,evalset)
        rows.append({'candidate':cand,'checkpoint':str(p),**z})
        pd.concat([prev,pd.DataFrame(rows)],ignore_index=True).to_csv(select_csv,index=False)
selection=pd.read_csv(select_csv)
pt=selection.pivot(index='candidate',columns='clean_or_shifted',values='mAP50')
pt['shift_score']=0.25*pt['tune_lowlight_s2']+0.50*pt['tune_lowlight_s3']+0.25*pt['tune_haze_s3']
base_clean=float(pt.loc['original_baseline','voc_tune'])
base_shift=float(pt.loc['original_baseline','shift_score'])
pt['clean_delta_vs_original']=pt['voc_tune']-base_clean
pt['shift_delta_vs_original']=pt['shift_score']-base_shift
pt['passes_clean_guard']=pt['clean_delta_vs_original'] >= -CFG['clean_guard_drop']
pt['pareto_better_than_baseline']=(pt['clean_delta_vs_original']>0)&(pt['shift_delta_vs_original']>0)
pt=pt.sort_values(['passes_clean_guard','shift_score'],ascending=False)
pt.to_csv(OUT/'tune_selection_scores.csv')
print(pt[['voc_tune','shift_score','clean_delta_vs_original','shift_delta_vs_original','passes_clean_guard','pareto_better_than_baseline']].round(4).to_string())
# Prefer a STRICT joint improvement (clean + robust). Only claim a win if both increase.
novel=pt.loc[[x for x in pt.index if x not in ('original_baseline','original_combined')]]
allowed=novel[novel['passes_clean_guard']]
EXPLORATORY=str(allowed['shift_score'].idxmax()) if len(allowed) else str(novel['shift_score'].idxmax())
pareto=novel[novel['pareto_better_than_baseline']]
if len(pareto):
    WINNER=str(pareto['shift_score'].idxmax())
    print('Joint clean+shift improvement on TUNING data; promoting:',WINNER)
else:
    WINNER='original_baseline'
    print('No strict Pareto winner on tuning. Original clean baseline retained.')
    print('We will STILL evaluate the best exploratory new arm transparently:',EXPLORATORY)
print('Selected using tuning data only:',WINNER)
(OUT/'selected_candidate.json').write_text(json.dumps({'candidate':WINNER,'checkpoint':str(candidate_paths[WINNER]),
                                                       'exploratory':EXPLORATORY,
                                                       'clean_guard_drop':CFG['clean_guard_drop'],
                                                       'selection_criterion':'strict positive clean and shifted tuning mAP50 deltas; tie break by shifted weighted score',
                                                       'selected_from':'VOC tune only'},indent=2))

## 9 — Build final held-out test datasets

This stage runs only **after** the candidate has been selected. It reconstructs the same VOC held-out split and tests synthetic weather at severity 1–3. If your RTTS/DAWN zips are accessible at the configured Drive paths, it reconstructs those real datasets using the fixed splits from the previous project. ExDark is optional if its annotated images are available locally.

**Do not adjust training choices after looking at test results.** If a real dataset is unavailable, its results are omitted rather than invented.

In [ ]:
export_imageset('voc_test',[(r,None) for r in test_recs])
SYN=test_recs[:CFG['syn_n']]
for kind in CFG['syn_types']:
    for sev in CFG['syn_severities']:
        name=f'voc_syn_{kind}_s{sev}'
        items=[(r,(kind,sev,[CFG['seed'],i,DEG_ID[kind],sev*10])) for i,r in enumerate(SYN)]
        export_imageset(name,items)

# Optional real datasets: parsed only for the final evaluation (never for checkpoint selection)
def extract_source(source,dest):
    if source is None or not Path(source).exists():return None
    src=Path(source)
    if src.is_dir():return src
    if (dest/'.extracted').exists():return dest
    dest.mkdir(parents=True,exist_ok=True)
    pending=[src]
    while pending:
        a=pending.pop()
        out=dest if a==src else a.parent/a.stem
        if zipfile.is_zipfile(a):
            with zipfile.ZipFile(a) as z:z.extractall(out)
        elif tarfile.is_tarfile(a):
            with tarfile.open(a) as t:t.extractall(out)
        else:continue
        if a!=src:a.unlink()
        pending += [p for p in out.rglob('*') if p.is_file() and p.suffix.lower() in ('.zip','.tar')]
    (dest/'.extracted').touch()
    return dest

def weather_condition(p):
    terms=re.split('[^a-z]+',str(p).lower())
    for k,co in [('fog','fog'),('haze','fog'),('mist','fog'),('rain','rain'),('snow','snow'),('sand','sand'),('dust','sand')]:
        if any(t.startswith(k) for t in terms):return co
    return 'unknown'

def parse_real_voc_folder(root,source):
    imgs=defaultdict(list)
    for ip in Path(root).rglob('*'):
        if ip.is_file() and ip.suffix.lower() in ('.jpg','.jpeg','.png','.bmp'):
            imgs[ip.stem.lower()].append(ip)
    recs=[]
    for xp in sorted(Path(root).rglob('*.xml')):
        if '__MACOSX' in xp.parts:continue
        try: r=parse_voc_xml(xp)
        except Exception:continue
        possible=imgs.get(xp.stem.lower(),[])
        if not possible:continue
        # Source archives sometimes reuse names across weather folders: prefer nearest paths.
        def prefix(p):
            a,b=p.parts,xp.parts;return len(list(__import__('itertools').takewhile(lambda z:z[0]==z[1],zip(a,b))))
        ip=max(possible,key=prefix)
        r.update(path=str(ip),id=str(ip.relative_to(root)),source=source,condition=weather_condition(ip))
        recs.append(r)
    return recs

def fixed_external_test(recs,name):
    # Matches the original notebook's Section 5: held out 15% for an unlabeled pool,
    # stratified by DAWN condition; the rest is test.
    rng=random.Random(CFG['seed']+17)
    groups=defaultdict(list)
    for rec in recs:
        g=rec['condition'] if name=='dawn' else 'all'
        groups[g].append(rec)
    test=[]
    for g in sorted(groups):
        records=groups[g][:];rng.shuffle(records)
        k=round(len(records)*0.15)
        test += records[k:]
    return test

REAL_TEST_NAMES=[]
if CFG['evaluate_real']:
    for name,key in [('rtts','rtts_source'),('dawn','dawn_source')]:
        src=CFG[key]
        root=extract_source(src,LOCAL_DATA/name)
        if not root:
            print(name.upper(),'not available:',src,'— skipping')
            continue
        recs=parse_real_voc_folder(root,name)
        if not recs:
            print(name.upper(),'has no paired VOC XML/images; skipped');continue
        external=fixed_external_test(recs,name)
        print(name.upper(),'total',len(recs),'held-out test',len(external))
        export_imageset(f'{name}_test',[(r,None) for r in external])
        REAL_TEST_NAMES.append(f'{name}_test')
        if name=='dawn':
            for cond in sorted(set(r['condition'] for r in external)):
                subset=[r for r in external if r['condition']==cond]
                if len(subset)>=20:
                    key=f'dawn_test_{cond}'
                    export_imageset(key,[(r,None) for r in subset]);REAL_TEST_NAMES.append(key)

    if CFG['exdark_source'] and Path(CFG['exdark_source']).exists():
        root=Path(CFG['exdark_source'])
        imap={p.name.lower():p for p in root.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')}
        splitmap={}
        cl=next(root.rglob('imageclasslist.txt'),None)
        if cl:
            for line in cl.read_text(errors='ignore').splitlines()[1:]:
                parts=line.split()
                if len(parts)>=5:
                    try:splitmap[parts[0].lower()]=int(parts[4])
                    except ValueError:pass
        recs=[]
        for ann in sorted(root.rglob('*.txt')):
            ip=imap.get(ann.stem.lower())
            if not ip:continue
            with Image.open(ip) as im:w,h=im.size
            objs=[]
            for line in ann.read_text(errors='ignore').splitlines():
                a=line.split()
                if len(a)<5 or a[0].startswith('%'):continue
                c=NAME_MAP.get(a[0].lower())
                if c is None:continue
                try:l,t,bw,bh=map(float,a[1:5])
                except ValueError:continue
                objs.append((c,l,t,l+bw,t+bh,0))
            recs.append(dict(path=str(ip),w=w,h=h,objs=objs,labels=sorted(set(o[0] for o in objs)),
                             id=ip.name,split=splitmap.get(ip.name.lower()),source='exdark'))
        if recs:
            if any(r['split'] is not None for r in recs):
                external=[r for r in recs if r['split']==3]
            else:
                rng=random.Random(17);rng.shuffle(recs);external=recs[round(len(recs)*0.15):]
            if external:
                export_imageset('exdark_test',[(r,None) for r in external]);REAL_TEST_NAMES.append('exdark_test')
                print('ExDark external tests',len(external))
print('Available real condition tests:',REAL_TEST_NAMES)

## 10 — Final blind comparison: original baseline, original combined, selected candidate

In [ ]:
# This is final evaluation — run ONCE after selection. Do not pick new hyperparameters using this table.
FINAL_MODELS={'original_baseline':ORIGINAL_BASE}
if ORIGINAL_COMB:FINAL_MODELS['original_combined']=ORIGINAL_COMB
FINAL_MODELS[EXPLORATORY]=candidate_paths[EXPLORATORY]
FINAL_MODELS[WINNER]=candidate_paths[WINNER]
FINAL_TESTS=['voc_test']+[f'voc_syn_{k}_s{s}' for k in CFG['syn_types'] for s in CFG['syn_severities']]+REAL_TEST_NAMES
final_csv=OUT/'heldout_detection_results.csv'
prev=pd.read_csv(final_csv) if final_csv.exists() else pd.DataFrame()
known=set(zip(prev['candidate'],prev['clean_or_shifted'])) if len(prev) else set()
rows=[]
for cand,p in FINAL_MODELS.items():
    for ev in FINAL_TESTS:
        if (cand,ev) in known:continue
        print('Held-out evaluation:',cand,ev)
        z=eval_one(p,ev)
        rows.append({'candidate':cand,'checkpoint':str(p),**z})
        pd.concat([prev,pd.DataFrame(rows)],ignore_index=True).to_csv(final_csv,index=False)
final=pd.read_csv(final_csv)
cols=['candidate','clean_or_shifted','mAP50','mAP5095','minority_AP50','minority_recall','majority_AP50','majority_recall']
print(final[final.clean_or_shifted.isin(['voc_test','voc_syn_lowlight_s3','voc_syn_haze_s3','rtts_test','dawn_test','exdark_test'])][cols].round(3).to_string(index=False))

## 11 — A reproducible verdict and plots for the poster

In [ ]:
final=pd.read_csv(OUT/'heldout_detection_results.csv')
pivot=final.pivot_table(index='candidate',columns='clean_or_shifted',values='mAP50',aggfunc='first')
base=pivot.loc['original_baseline']
compare=pivot.sub(base,axis=1)
compare.to_csv(OUT/'mAP50_delta_vs_original.csv')
print('Held-out mAP@0.5 DIFFERENCES relative to original baseline (positive means improvement):')
print(compare.round(4).to_string())
if WINNER in pivot.index and WINNER!='original_baseline':
    clean_delta=float(compare.loc[WINNER,'voc_test'])
    shifts=[c for c in pivot.columns if c.startswith('voc_syn_')]
    robust_delta=float(compare.loc[WINNER,shifts].mean())
    improved_both=clean_delta>0 and robust_delta>0
    print(f'\nSelected {WINNER}: VOC CLEAN Δ={clean_delta:+.4f}, synthetic mean Δ={robust_delta:+.4f}')
    print('Better on BOTH clean and average synthetic?:', improved_both)
    if not improved_both:
        print('No joint improvement demonstrated. Report limitations rather than claiming success.')
else:
    print('Original baseline retained: no strict joint improvement was found on tuning.')
    if EXPLORATORY in pivot.index:
        shifts=[c for c in pivot.columns if c.startswith('voc_syn_')]
        print(f'Exploratory {EXPLORATORY}: clean Δ={compare.loc[EXPLORATORY,"voc_test"]:+.4f}, synthetic mean Δ={compare.loc[EXPLORATORY,shifts].mean():+.4f}.')

# Plot a digestible performance panel for the poster.
sets=['voc_test','voc_syn_lowlight_s3','voc_syn_haze_s3','voc_syn_rain_s3','voc_syn_snow_s3','voc_syn_sandstorm_s3',
      'rtts_test','dawn_test','exdark_test']
sets=[s for s in sets if s in pivot.columns]
if sets:
    models=[x for x in ['original_baseline','original_combined',EXPLORATORY,WINNER] if x in pivot.index]
    models=list(dict.fromkeys(models))
    x=np.arange(len(sets));width=0.78/len(models)
    fig,ax=plt.subplots(figsize=(max(12,len(sets)*1.5),5))
    for i,m in enumerate(models):
        ax.bar(x+(i-(len(models)-1)/2)*width,pivot.loc[m,sets],width,label=m)
    ax.set_xticks(x,sets,rotation=30,ha='right')
    ax.set_ylabel('Detection mAP@0.5');ax.set_ylim(0,1)
    ax.set_title('Held-out clean and shifted performance (same evaluation sets)')
    ax.legend();fig.tight_layout()
    fig.savefig(OUT/'novelty_comparison.png',dpi=200);plt.show()

## 12 — Failure type analysis: missed objects, wrong labels, and inaccurate bounding boxes

In [ ]:
# Fixed threshold for *diagnostics*, not Ultralytics AP calculation.
# Qualitative samples are taken deterministically, not handpicked to make a model look better.
def xyxy_iou(a,b):
    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    a1=max(0,a[2]-a[0])*max(0,a[3]-a[1]);a2=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(a1+a2-inter+1e-9)

def groundtruth(image,labelfile):
    with Image.open(image) as im:w,h=im.size
    result=[]
    for line in Path(labelfile).read_text().splitlines():
        a=line.split()
        if len(a)!=5:continue
        c,x,y,bw,bh=map(float,a)
        result.append((int(c),np.array([(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h])))
    return result

def predict_for_image(model,image,conf=0.15):
    r=model.predict(source=str(image),conf=conf,iou=0.6,verbose=False,imgsz=CFG['imgsz'],device=0 if CUDA else 'cpu')[0]
    return [(int(c),b,float(s)) for c,b,s in zip(r.boxes.cls.int().cpu().numpy(),
                                                r.boxes.xyxy.cpu().numpy(),r.boxes.conf.cpu().numpy())]

def diagnose(gt,preds):
    counts=Counter();used=set()
    # Match confident correct-class boxes at IoU>=0.5 one-to-one first.
    matches=[]
    for i,(c,b) in enumerate(gt):
        for j,(pc,pb,score) in enumerate(preds):
            if c==pc:
                v=xyxy_iou(b,pb)
                if v>=0.5:matches.append((v,i,j))
    found=set()
    for v,i,j in sorted(matches,reverse=True):
        if i not in found and j not in used:
            found.add(i);used.add(j);counts['correct']+=1
    for i,(c,b) in enumerate(gt):
        if i in found:continue
        best_same=max([xyxy_iou(b,pb) for j,(pc,pb,_) in enumerate(preds) if pc==c and j not in used] or [0.0])
        best_wrong=max([xyxy_iou(b,pb) for j,(pc,pb,_) in enumerate(preds) if pc!=c and j not in used] or [0.0])
        if best_wrong>=0.5: counts['wrong_class']+=1
        elif best_same>=0.10: counts['poor_box']+=1
        else: counts['missed']+=1
    return counts

# A modest fixed set of examples focusing on low light + clean conditions.
FAILURE_SETS=['voc_test','voc_syn_lowlight_s3']
MODELS_FOR_DIAG={'original_baseline':YOLO(str(ORIGINAL_BASE)),EXPLORATORY:YOLO(str(candidate_paths[EXPLORATORY]))}
errors=[]
from PIL import ImageDraw

def draw_pred(image,preds,gt,header):
    image=Image.open(image).convert('RGB');d=ImageDraw.Draw(image)
    for c,box in gt:
        d.rectangle(list(map(float,box)),outline='#36D399',width=3)
    for c,b,conf in preds:
        d.rectangle(list(map(float,b)),outline='#F15B50',width=2)
        d.text((float(b[0]),max(0,float(b[1])-13)),f'{CLASSES[c]} {conf:.2f}',fill='#F15B50')
    d.text((10,10),header,fill='yellow')
    return image

for split in FAILURE_SETS:
    folder=YDATA/split
    paths=sorted((folder/'images').glob('*'))
    picked=[]
    for p in paths:
        lab=folder/'labels'/f'{p.stem}.txt'
        gt=groundtruth(p,lab)
        if any(c>=5 for c,_ in gt):picked.append(p)
        if len(picked)>=12:break
    for ix,p in enumerate(picked):
        lab=folder/'labels'/f'{p.stem}.txt';gt=groundtruth(p,lab)
        all_pred={}
        for name,m in MODELS_FOR_DIAG.items():
            pr=predict_for_image(m,p);all_pred[name]=pr
            counts=diagnose(gt,pr)
            errors.append({'dataset':split,'image':p.name,'model':name,'n_groundtruth':len(gt),**counts})
        if ix<6:
            panels=[draw_pred(p,all_pred[n],gt,n) for n in MODELS_FOR_DIAG]
            W=min(800,panels[0].width)
            panels=[q.resize((W,round(q.height*W/q.width))) for q in panels]
            merged=Image.new('RGB',(sum(q.width for q in panels),max(q.height for q in panels)))
            cur=0
            for q in panels:merged.paste(q,(cur,0));cur+=q.width
            merged.save(OUT/f'qualitative_{split}_{ix:02d}.jpg',quality=92)
# Counter omits keys whose counts are zero. Make every diagnostic column explicit.
count_cols = ['correct', 'wrong_class', 'poor_box', 'missed']
err = pd.DataFrame(errors)
for name in count_cols:
    if name not in err.columns:
        err[name] = 0
err[count_cols] = err[count_cols].fillna(0).astype(int)
err.to_csv(OUT/'sample_failure_analysis.csv', index=False)
if not err.empty:
    print('DIAGNOSTIC COUNTS (fixed conf 0.15; sampled minority-positive images; not formal full-test recall):')
    print(err.groupby(['dataset', 'model'])[count_cols].sum().to_string())
else:
    print('No minority-positive images were sampled; no diagnostics to summarize.')
print('Saved qualitative comparisons to', OUT)

## 13 — What can we conclude?

**Required reporting rules:**

1. Claim a clean-image improvement only if held-out VOC clean mAP@0.5 actually increases.
2. Claim better robustness only for the specific shifted datasets/conditions that improve, including majority/minority recall.
3. The three novel arms are controlled experiments; no one arm is automatically the winner.
4. Real weather datasets can have different class coverage. Never compare their raw mAP across datasets as if they annotate identical categories.
5. Severe low-light synthetic images may become information-limited. No architecture can recover details genuinely absent from the pixels.
6. The original object detection system ignores boxes of classes outside the selected ten. This is a limitation of the task-specific subset and may produce background-label noise.
7. Figures, `.csv`s, training logs, model checkpoints, and a candidate selection summary are stored in your Google Drive novelty folder. Use **verified** values from the generated files in the poster and report.

**For the poster:** the useful scientific narrative is *pre-existing clean/shift trade-off → matched-budget clean-anchor ablation → minority- and weather-wise analysis → successes and failures*. The novelty is the targeted combination and validation protocol, **not** a claim to have invented class-balanced augmentation or YOLOv8 itself.